# Custom-Optimizers: test on a Colab T4
First: **Runtime > Change runtime type > T4 GPU**. Then run the cells top to bottom.

## Step 1: check the GPU

In [ ]:
import sys, torch
print("python", sys.version.split()[0])
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU. Runtime > Change runtime type > T4 GPU, then re-run from the top.")

## Step 2: get the code (pick ONE option)
**Option A (works for private repos): upload the zip.**

In [ ]:
from google.colab import files
import os, zipfile
uploaded = files.upload()          # choose custom-optimizers.zip
zip_name = next(iter(uploaded))
os.makedirs("/content/custom-optimizers", exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/custom-optimizers")
%cd /content/custom-optimizers
!ls

**Option B (public repo): clone it.** Replace YOUR_USERNAME, and skip Option A.

In [ ]:
# !git clone https://github.com/YOUR_USERNAME/Custom-Optimizers.git /content/custom-optimizers
# %cd /content/custom-optimizers
# !ls

## Step 3: install the few missing packages (Colab already has torch and jax)

In [ ]:
!pip install -q pytest pyyaml scikit-learn matplotlib

## Step 4: run the full test suite (expect 65 passed)

In [ ]:
!JAX_PLATFORMS=cpu python -m pytest -q

## Step 5: run the PyTorch engine on the GPU and compare it with the NumPy reference

In [ ]:
import sys, numpy as np, torch
sys.path.insert(0, "/content/custom-optimizers")
from src.environment import market_from_yaml, DeterministicBatchSampler
from src.torch_optimizer import CoordinateSVRG
from src.reference_numpy import ReferenceConfig, ReferenceSVRG

dev = "cuda"
ds = market_from_yaml("configs/stochastic_regime.yaml")
X = torch.as_tensor(ds.features, dtype=torch.float32, device=dev)
y = torch.as_tensor(ds.targets, dtype=torch.float32, device=dev)
sampler = DeterministicBatchSampler(ds.n_samples, 64, ds.config.seed)
chunks = sampler.full_pass_chunks(8)
w = torch.nn.Parameter(torch.zeros(ds.n_features, device=dev))

def loss_on(rows):
    idx = torch.as_tensor(rows, device=dev)
    r = X[idx] @ w - y[idx]
    return 0.5 * (r * r).mean()

batch_closure = lambda batch_id: loss_on(sampler.batch(batch_id))
chunk_closure = lambda chunk_id: loss_on(chunks[chunk_id])

opt = CoordinateSVRG([w], lr=0.02, snapshot_interval=64)
ref = ReferenceSVRG(ReferenceConfig(lr=0.02, snapshot_interval=64), ds.n_features, dtype=np.float32)

Xn, yn = ds.features.astype(np.float32), ds.targets.astype(np.float32)
grad = lambda wv, rows: Xn[rows].T @ (Xn[rows] @ wv - yn[rows]) / len(rows)
full_grad = lambda wv: Xn.T @ (Xn @ wv - yn) / len(yn)

w_ref = np.zeros(ds.n_features, dtype=np.float32)
max_err = 0.0
for k in range(100):
    if opt.needs_snapshot:
        opt.refresh_snapshot(chunk_closure, num_chunks=8)
    opt.step(batch_closure)
    if ref.needs_snapshot:
        ref.refresh_snapshot(w_ref, full_grad(w_ref))
    rows = sampler.batch(k)
    w_ref = ref.step(w_ref, grad(w_ref, rows), grad(ref.snapshot, rows)).astype(np.float32)
    max_err = max(max_err, float(np.max(np.abs(w.detach().cpu().numpy() - w_ref))))

print("weights live on:", w.device)
print("max |GPU engine - NumPy reference| over 100 steps:", max_err)
print("last telemetry:", {k: round(v, 6) for k, v in opt.last_stats.items() if k in ("step", "loss", "snapshot_distance", "step_us")})
print("PASS" if max_err < 1e-3 else "CHECK: difference is larger than expected, send me this output")

## Step 6: GPU benchmark (SVRG vs Adam vs SGD+momentum, same data and batches)

In [ ]:
!python -m benchmarks.compare_optimizers --device cuda --wandb-mode disabled --noise-regimes volatile --seeds 1

## Step 7 (optional): smoke-test the whole experiment suite with tiny settings

In [ ]:
!python -m experiments.run_experiments --quick --out /tmp/out --figures /tmp/figs

## Step 8: walk-forward study on REAL market data (needs internet; takes a few minutes)
Downloads daily prices for 10 ETFs from Yahoo Finance and compares OLS, ridge, budget-limited Adam, budget-limited Jacobi SVRG and 21-day momentum out of sample. **Expect small or null results**: daily returns are barely predictable, and the placebo run (circularly shifted targets) shows what noise looks like. If the download fails, put one CSV per asset (Date, Close, Volume) in a folder and use `--csv-dir`.

In [ ]:
!pip install -q yfinance

In [ ]:
!python -m experiments.real_market_study --tickers SPY QQQ IWM EFA EEM TLT GLD XLF XLE XLK --start 2008-01-01 --end 2025-12-31 --placebo --out /content/real_market

In [ ]:
from IPython.display import Image, display
display(Image("/content/real_market/cumulative_pnl.png"))

## Step 9: GPU timing study (answers 'is there any wall-clock win?')
Times direct solves, Jacobi SVRG (the real PyTorch engine) and Adam on the GPU for growing sizes, plus telemetry-on versus telemetry-off step cost and peak memory. Takes a few minutes. Needs the T4 runtime; if memory runs out, drop the biggest size.

In [ ]:
!python -m benchmarks.timing_study --sizes 100000x256 200000x1024 200000x2048

In [ ]:
print(open("results/timing/report.md").read())